# Rezervacija hotela z vmesnikom za prioritete članov

Ta zvezek prikazuje **vmesnik na podlagi funkcij** z uporabo Microsoft Agent Framework. Nadgrajujemo primer znotraj pogojnega delovnega toka z dodatno vmesno plastjo, ki članom s privilegiji daje posebne pravice.

## Kaj se boste naučili:
1. **Vmesnik na podlagi funkcij**: Prestrezanje in spreminjanje rezultatov funkcij
2. **Dostop do konteksta**: Branje in spreminjanje `context.result` po izvedbi
3. **Implementacija poslovne logike**: Koristi za člane s prioritetami
4. **Prepis rezultata**: Spreminjanje zaključkov funkcije glede na status uporabnika
5. **Isti delovni tok, različni rezultati**: Spremembe vedenja, ki jih povzroča vmesnik

## Arhitektura delovnega toka z vmesnikom:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Ključna razlika od pogojnega delovnega toka:

**Brez vmesnika** (14-conditional-workflow.ipynb):
- Pariz nima sob → Posredovanje k alternative_agent

**Z vmesnikom** (ta zvezek):
- Navaden uporabnik + Pariz → Ni sob → Posredovanje k alternative_agent
- Prioritetni uporabnik + Pariz → 🌟 Vmesnik prepiše! → Na voljo → Posredovanje k booking_agent

## Zahteve:
- Nameščen Microsoft Agent Framework
- Razumevanje pogojnih delovnih tokov (glej 14-conditional-workflow.ipynb)
- GitHub žeton ali OpenAI API ključ
- Osnovno razumevanje vzorcev vmesnikov


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## 1. korak: Določite Pydantic modele za strukturirane izhode

Ti modeli določajo **shemo**, ki jo bodo agenti vrnili. Dodali smo polje `priority_override`, da sledimo, kdaj vmesna programska oprema spremeni rezultat razpoložljivosti.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Korak 2: Določitev baze podatkov članov s prednostjo

Za ta prikaz bomo simulirali bazo podatkov članov s prednostjo. V produkciji bi to poizvedovalo v pravi bazi podatkov ali API-ju.

**Člani s prednostjo:**
- `alice@example.com` - VIP član
- `bob@example.com` - Premium član  
- `priority_user` - Testni račun


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Korak 3: Ustvarite orodje za rezervacijo hotela

Enako kot pogojni potek dela, vendar ga bo zdaj prestregla vmesna programska oprema!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Korak 4: 🌟 Ustvarite vmesnik za preverjanje prioritet (KLJUČNA FUNKCIJA!)

To je **osnovna funkcionalnost** tega zvezka. Vmesnik:

1. **Zajame** klic funkcije hotel_booking
2. **Izvede** funkcijo običajno s klicem `next(context)`
3. **Pregleda** rezultat v `context.result`
4. **Prepiše** rezultat, če je uporabnik prioriteta in ni razpoložljivih sob
5. **Vrne** spremenjeni rezultat nazaj agentu

**Ključni vzorec:**
```python
async def my_middleware(context, next):
    await next(context)  # Izvedi funkcijo
    # Zdaj kontekst.result vsebuje izhod funkcije
    if some_condition:
        context.result = new_value  # Prepiši!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## 5. korak: Določite funkcije pogojev za usmerjanje

Enake funkcije pogojev kot pri pogojnem delovnem toku – pregledajo strukturiran rezultat za določitev usmerjanja.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Korak 6: Ustvarite po meri prilagojen prikazovalnik izvršilca

Enak izvršilec kot prej - prikazuje končni izhod poteka dela.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Korak 7: Naložite spremenljivke okolja

Konfigurirajte LLM odjemalca (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## Korak 8: Ustvarjanje AI agentov z vmesno programsko opremo

**KLJUČNA RAZLIKA:** Pri ustvarjanju availability_agent posredujemo parameter `middleware`!

Tako vbrizgamo priority_check_middleware v cevovod klicov funkcij agenta.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## Korak 9: Sestavite potek dela

Enaka struktura poteka dela kot prej - pogojno usmerjanje glede na razpoložljivost.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Korak 10: Testni primer 1 - Redni uporabnik v Parizu (Brez preglasitve)

Redni uporabnik poskuša rezervirati Pariz → Brez sob → Preusmeri na alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Korak 11: Testni primer 2 - 🌟 Prioritetni uporabnik v Parizu (S PREKLOPOM!)

Prioritetni član poskuša rezervirati Pariz → Sprva ni sob → 🌟 Middleware prekliče to! → Usmerjeno na booking_agent

**To je ključna demonstracija moči middleware!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Korak 12: Testni primer 3 - Prioritetni uporabnik v Stockholmu (že na voljo)

Prioritetni uporabnik poskuša Stockholm → Sobe so na voljo → Ni potrebno preglasiti → Preusmerjeno na booking_agent

To kaže, da vmesniška programska oprema deluje le, ko je potrebno!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Ključne ugotovitve in koncepti vmesnika (middleware)

### ✅ Kaj ste se naučili:

#### **1. Vzorec vmesnika, ki temelji na funkcijah**

Vmesnik prestreza klice funkcij z uporabo preproste asinhrone funkcije:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Pred izvajanjem funkcije
    print("Intercepting...")
    
    # Izvedi funkcijo
    await next(context)
    
    # Po izvedbi funkcije - preveri rezultat
    if context.result:
        # Po potrebi spremeni rezultat
        context.result = modified_value
```

#### **2. Dostop do konteksta in sprememba rezultata**

- `context.function` - Dostop do klicane funkcije
- `context.arguments` - Branje argumentov funkcije
- `context.kwargs` - Dostop do dodatnih parametrov
- `await next(context)` - Izvedba funkcije
- `context.result` - Branje/sprememba izhoda funkcije

#### **3. Implementacija poslovne logike**

Naš vmesnik izvaja prednostne koristi za člane:
- **Redni uporabniki**: Brez sprememb, standardno delovanje
- **Prednostni uporabniki**: Prepiše "ni razpoložljivosti" → "na voljo"
- **Pogojna logika**: Prepiše le, kadar je potrebno

#### **4. Enak potek dela, različni rezultati**

Moč vmesnika:
- ✅ Brez sprememb strukture poteka dela
- ✅ Brez sprememb funkcije orodja
- ✅ Brez sprememb pogojne logike usmerjanja
- ✅ Samo vmesnik → Različno vedenje!

### 🚀 Uporabe v resničnem svetu:

1. **VIP/Premium funkcije**
   - Prepiše omejitve hitrosti za premium uporabnike
   - Omogoči prednostni dostop do virov
   - Dinamično odklepanje premium funkcij

2. **A/B testiranje**
   - Usmerja uporabnike na različne implementacije
   - Testira nove funkcije z določenimi uporabniki
   - Postopni uvodi funkcij

3. **Varnost in skladnost**
   - Revizija klicev funkcij
   - Blokira občutljive operacije
   - Uvede poslovna pravila

4. **Optimizacija zmogljivosti**
   - Predpomni rezultate za določene uporabnike
   - Preskoči drage operacije, kadar je mogoče
   - Dinamična dodelitev virov

5. **Ravnanje z napakami in ponovni poskusi**
   - Ujame in elegantno obdela napake
   - Izvede logiko ponovnega poskusa
   - Uporabi alternativne implementacije

6. **Dnevnik in nadzor**
   - Sledi časom izvajanja funkcij
   - Zabeleži parametre in rezultate
   - Nadzira vzorce uporabe

### 🔑 Ključne razlike glede na dekoratorje:

| Značilnost | Dekorator | Vmesnik (Middleware) |
|---------|-----------|------------|
| **Obseg** | Ena sama funkcija | Vse funkcije v agentu |
| **Prilagodljivost** | Fiksna ob definiciji | Dinamična ob izvajanju |
| **Kontekst** | Omejen | Celoten kontekst agenta |
| **Sestava** | Več dekoratorjev | Vmesniški cevovod |
| **Zavedanje agenta** | Ne | Da (dostop do stanja agenta) |

### 📚 Kdaj uporabiti vmesnik:

✅ **Uporabite vmesnik, ko:**
- Potrebujete spremembo vedenja glede na stanje uporabnika/seje
- Želite uporabiti logiko na več funkcijah
- Potrebujete dostop do konteksta na ravni agenta
- Uvajate prečne skrbi (dnevnik, avtentikacija itd.)

❌ **Ne uporabljajte vmesnika, ko:**
- Preprosta validacija vnosa (uporabite Pydantic)
- Funkcijsko specifična logika (naj ostane v funkciji)
- Enkratne spremembe (spremenite samo funkcijo)

### 🎓 Napredni vzorci:

```python
# Več vmesnih stopenj (pomemben je vrstni red izvajanja!)
middleware=[
    logging_middleware,      # Najprej zapisi
    auth_middleware,         # Nato preveri avtentikacijo
    cache_middleware,        # Nato preveri predpomnilnik
    rate_limit_middleware,   # Nato omeji hitrost
    priority_check_middleware  # Nazadnje preveri prednost
]

# Pogojno izvajanje vmesnih stopenj
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Spremeni rezultat
    else:
        # Popolnoma preskoči izvajanje
        context.result = cached_value
```

### 🔗 Sorodni koncepti:

- **Agent Middleware**: Prestreže klice agent.run()
- **Funkcijski Middleware**: Prestreže klice funkcij orodja (to smo uporabili!)
- **Middleware pipeline**: Veriga vmesnikov, ki se izvajajo zaporedno
- **Propagacija konteksta**: Prenos stanja skozi vmesniško verigo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
